# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
rows_dupes = int(df.duplicated().sum())
df = df.drop_duplicates()

log('duplicates', 'dropped duplicate rows', rows_dupes)

[duplicates] dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
rows_price_bad = df['price'].astype(str).str.contains('$', regex=False).sum()
df['price'] = (df['price'].astype(str)
                   .str.replace('$', '', regex=False)
                   .str.strip()
                   .astype(float))

log('fixing_price', 'made all price as floats', rows_price_bad)

[fixing_price] made all price as floats (124 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
rows_qty_bad = (df['qty'].isna().sum()) + ((df['qty'] <= 0).sum())
df = df.dropna(subset=['qty'])
df = df[df['qty'] > 0]

log('fixing_qty', 'got rid of negative or missing values in qty', rows_qty_bad)

[fixing_qty] got rid of negative or missing values in qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())

ITEM_MAP = {'Foam Finger' : 'Foam Finger',
            'foam finger' : 'Foam Finger',
            'Rain Poncho' : 'Rain Poncho',
            'rain poncho' : 'Rain Poncho',
            'Cheeseburger' : 'Cheeseburger',
            'cheeseburger' : 'cheeseburger'}

before_item = df['category'].copy()
df['item'] = df['item'].map(ITEM_MAP)
rows_collapsed = int((before_item != df['item']).sum())

print(df['item'].value_counts())

log('fixing item', 'merge 6 items into 3', rows_collapsed)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    43
Name: count, dtype: int64
[fixing item] merge 6 items into 3 (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())
ITEM_MAP = {'Food' : 'Food',
            'food' : 'Food',
            'rain-gear' : 'Rain Gear',
            'RainGear' : 'Rain Gear',
            'Merch' : 'Merch',
            'Apparel' : 'Apparel'}

before_cat = df['category'].copy()
df['category'] = df['category'].map(ITEM_MAP)
rows_collapsed_cat = int((before_cat != df['category']).sum())

print(df['category'].value_counts())

log('fixing category', 'merge 6 items into 4', rows_collapsed_cat)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
category
Food         95
Rain Gear    86
Merch        51
Apparel      43
Name: count, dtype: int64
[fixing category] merge 6 items into 4 (130 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
assert df['item'].nunique() == 3
# TODO: assert something about category
assert df['category'].nunique() == 4
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = ((df['qty']) * (df['price'])).round(2)
print(df.groupby('category')['revenue'].sum().sort_values(ascending=False))

total = df['revenue'].sum()
print(total)


category
Food         1656.0
Rain Gear    1512.0
Merch         856.5
Apparel       715.5
Name: revenue, dtype: float64
4740.0


**What I would tell the vendor:** Stock more food.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped duplicate rows,15
1,fixing_price,made all price as floats,124
2,fixing_qty,got rid of negative or missing values in qty,25
3,fixing item,merge 6 items into 3,275
4,fixing category,merge 6 items into 4,130


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Step 3; revenue before = 4594.5, revenue after = 4740.0

b) Merch and Apparel could have been merged; I decided against it because Merch is a more general term and could apply to items besides clothing, which is exclusively represented by the category Apparel. This wouldn't have affected the total revenue because it would only have affected the number of categories, not the 'qty' or 'price' variables.